# 1. Import

In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 2. Load Data

In [8]:
df = pd.read_csv("../data/diabetic_data_cleanedv2.csv")
print(f"Loaded cleaned data: {df.shape}")

TARGET_COL = "readmitted"
GROUP_COL = "patient_nbr"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

Loaded cleaned data: (99340, 47)


# 3. Encode Categorical Features

In [9]:
# แยกคอลัมน์เป็น categorical (dtype object) กับ numeric
# หมายเหตุ: patient_nbr เป็นตัวเลขแต่ไม่ใช่ feature เชิงทำนาย (เป็นแค่ ID ที่
# ใช้ group ตอน split) จึงกันออกจากทั้งสองกลุ่มไว้ก่อน แล้วค่อยตัดออกจาก X
# ทีหลังสุด หลัง split เสร็จ
feature_cols = [c for c in X.columns if c != GROUP_COL]
categorical_cols = X[feature_cols].select_dtypes(include=["object", "str"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")

Categorical columns (32): ['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']
Numeric columns (13): ['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'total_prior_visits', 'had_prior_inpatient']


In [10]:
# age เป็น ordinal (ช่วงอายุที่มีลำดับ) -> map เป็นตัวเลขแทนตำแหน่งกลางช่วง
# แทนที่จะ one-hot encode เพื่อรักษาความสัมพันธ์เชิงลำดับไว้
if "age" in categorical_cols:
    age_map = {
        "[0-10)": 5, "[10-20)": 15, "[20-30)": 25, "[30-40)": 35,
        "[40-50)": 45, "[50-60)": 55, "[60-70)": 65, "[70-80)": 75,
        "[80-90)": 85, "[90-100)": 95,
    }
    X["age"] = X["age"].map(age_map)
    categorical_cols.remove("age")
    numeric_cols.append("age")

# ที่เหลือ (เพศ, เชื้อชาติ, medical_specialty, กลุ่มยา, diag_1-3 ฯลฯ) ใช้
# Label Encoding — เพียงพอสำหรับ tree-based models (RF/AdaBoost/XGBoost) และ
# เร็วกว่า one-hot ที่จะทำให้มิติข้อมูลระเบิดเมื่อ cardinality ปานกลาง-สูง
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# 4. Train / Test Split (แก้ไข — GroupShuffleSplit)

**เดิม** ใช้ `train_test_split` แบบสุ่มรายแถวธรรมดา ซึ่งจะทำให้ผู้ป่วยคนเดียวกัน (ที่ตอนนี้มีหลาย encounter ต่อคนแล้ว เพราะ 02_preprocessing ไม่ตัดทิ้งแล้ว) กระจายไปอยู่ทั้งใน train และ test set — โมเดลจะ "จำ" ผู้ป่วยจาก train แล้วไปเจอ encounter อื่นของคนเดิมใน test เป็นการรั่วไหลของข้อมูล (data leakage) ทำให้ตัวเลข ประเมินผลดูดีเกินจริง

**แก้ไข**: ใช้ `GroupShuffleSplit` แบ่งตาม `patient_nbr` แทน รับประกันว่าผู้ป่วยแต่ละคน จะอยู่ใน train หรือ test อย่างใดอย่างหนึ่งเท่านั้น ไม่มีทางไปโผล่ทั้งสองฝั่ง

In [11]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df[GROUP_COL]))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# ตัด patient_nbr ออกจาก X หลัง split เสร็จแล้ว (ใช้ประโยชน์หมดแล้ว ไม่ใช่
# feature เชิงทำนาย)
X_train.drop(columns=[GROUP_COL], inplace=True)
X_test.drop(columns=[GROUP_COL], inplace=True)

# ยืนยันว่าไม่มีผู้ป่วยคนไหนอยู่ทั้ง train และ test
train_patients = set(df.loc[train_idx, GROUP_COL])
test_patients = set(df.loc[test_idx, GROUP_COL])
overlap = train_patients & test_patients
print(f"Patients overlapping between train/test: {len(overlap)} (ต้องเป็น 0)")

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Train target distribution:")
print(y_train.value_counts(normalize=True))

Patients overlapping between train/test: 0 (ต้องเป็น 0)

Train shape: (79567, 45), Test shape: (19773, 45)
Train target distribution:
readmitted
0    0.886008
1    0.113992
Name: proportion, dtype: float64


# 5. Scale Numeric Features

In [12]:
# Logistic Regression (meta-learner ของ stacking) ไวต่อ scale ของฟีเจอร์
# ส่วน RF/AdaBoost/XGBoost ไม่จำเป็นต้อง scale แต่การ scale ไม่ทำให้แย่ลง
# จึง fit scaler บน train แล้ว apply กับทั้ง train/test (กัน data leakage)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# 6. Save Everything

In [13]:
joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "label_encoders": label_encoders,
        "scaler": scaler,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "feature_names": X_train.columns.tolist(),
    },
    "../data/prepared_data.pkl",
)

print("\nSaved prepared data & preprocessing objects to: prepared_data.pkl")


Saved prepared data & preprocessing objects to: prepared_data.pkl
